In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [8]:
import json
from pydantic import ValidationError
import cv2

In [3]:
from src.schemas.coco_dataset import COCODataset

In [4]:
dataset_dir = Path("datasets/minecraft")

In [5]:
def check_annotation_structure(path: Path) -> bool:
    with open(path, mode="r") as file:
        data = json.load(file)
        try:
            COCODataset(**data)
            return True
        except Exception as e:
            raise e 
        

In [12]:
import json
import os
from pathlib import Path
import cv2

def check_annotations_and_images(data_dir: Path) -> bool:
    image_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.gif', '.svg', '.bmp', '.tiff', '.heic', '.avif')
    
    annotations_path = data_dir / "annotations.json"
    if not annotations_path.is_file():
        raise FileNotFoundError(f"Файл аннотаций не найден по пути: {annotations_path}")

    valid_images = set()
    for f in os.listdir(data_dir):
        if f.lower().endswith(image_extensions):
            full_path = data_dir / f
            
            img = cv2.imread(str(full_path))
            if img is not None:
                valid_images.add(f)
            else:
                print(f"⚠️ Файл поврежден или не читается: {f}")
        
    with open(annotations_path, mode="r", encoding="utf-8") as f:
        data = json.load(f)
    
    annotated_images = set([img_info["file_name"] for img_info in data.get("images", [])])
    
    diff_images_minus_ann = valid_images - annotated_images
    diff_ann_minus_images = annotated_images - valid_images
    
    if diff_images_minus_ann or diff_ann_minus_images:
        if diff_images_minus_ann:
            print("Картинки есть на диске, но отсутствуют в JSON (Images - Annotations):")
            print(diff_images_minus_ann)
            print("-" * 30)
        if diff_ann_minus_images:
            print("Картинки заявлены в JSON, но физически отсутствуют на диске/повреждены (Annotations - Images):")
            print(diff_ann_minus_images)
            print("-" * 30)
        return False
    else:
        print("✅ Все файлы и аннотации идеально совпадают!")
        return True


In [ ]:
dirs = ["train", "valid", "test"]

for dir in dirs:
    print(f"Проверка аннотаций в {dir}")
    try:
        checked = check_annotation_structure(dataset_dir / dir / "annotations.json")
    except ValidationError as e:
        print("❌ ОШИБКА ВАЛИДАЦИИ СТРУКТУРЫ COCO!")
        print(f"Всего обнаружено нестыковок: {e.error_count()}\n")
        print("--- ТОП-5 ПРОБЛЕМНЫХ МЕСТ В ДАННЫХ ---")
        
        for i, error in enumerate(e.errors()[:5]):
            path_to_error = " -> ".join(str(loc) for loc in error['loc'])
            error_type = error['type']
            error_msg = error['msg']
        
            bad_value = error.get('input', 'Не удалось определить')
            
            print(f"Ошибка #{i+1}:")
            print(f"  📍 Где искать: {path_to_error}")
            print(f"  📝 Что не так: {error_msg} (тип ошибки: {error_type})")
            print(f"  ❌ Что там записано сейчас: {bad_value}")
            print("-" * 40)
    print(f"Проверено {dir}")
print("Все ок!")

Проверка аннотаций в train
Проверено train
Проверка аннотаций в valid
Проверено valid
Проверка аннотаций в test
Проверено test
Все ок!


In [14]:
for dir in dirs:
    print(f"Проверка {dir}")
    check_annotations_and_images(dataset_dir / dir)

Проверка train
✅ Все файлы и аннотации идеально совпадают!
Проверка valid
✅ Все файлы и аннотации идеально совпадают!
Проверка test
✅ Все файлы и аннотации идеально совпадают!
